#Movie Dataset Analysis using PySpark

This notebook downloads MovieLens 25M (about 25 million ratings), loads it with PySpark, performs basic analysis, and saves CSV results.


## 1. Install PySpark

In [ ]:
!pip -q install pyspark

## 2. Download and extract the dataset

In [ ]:
import os
import zipfile

url = "https://files.grouplens.org/datasets/movielens/ml-25m.zip"
zip_path = "/content/ml-25m.zip"
data_dir = "/content/ml-25m"

if not os.path.exists(data_dir):
    print("Downloading MovieLens 25M. This may take a few minutes...")
    # Use wget with --no-check-certificate to bypass the SSL validation issue
    os.system(f'wget --no-check-certificate "{url}" -O "{zip_path}"')

    if os.path.exists(zip_path):
        print("Extracting dataset...")
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall("/content")
    else:
        print("Download failed.")
else:
    print("Dataset already extracted.")

print("Dataset folder ready:", data_dir)
if os.path.exists(data_dir):
    print(os.listdir(data_dir))

Extracting dataset...
Dataset folder ready: /content/ml-25m
['genome-tags.csv', 'genome-scores.csv', 'tags.csv', 'movies.csv', 'README.txt', 'ratings.csv', 'links.csv']


## 3. Start Spark and load CSV files

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, desc, split, explode

spark = (SparkSession.builder
         .appName("SimpleMovieAnalysis")
         .master("local[*]")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

movies = spark.read.option("header", True).option("inferSchema", True).csv(
    "/content/ml-25m/movies.csv"
)
ratings = spark.read.option("header", True).option("inferSchema", True).csv(
    "/content/ml-25m/ratings.csv"
)

print("Movies:", movies.count())
print("Ratings:", ratings.count())
movies.show(5, truncate=False)
ratings.show(5)

Movies: 62423
Ratings: 25000095
+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |
|5      |Father of the Bride Part II (1995)|Comedy                                     |
+-------+----------------------------------+-------------------------------------------+
only showing top 5 rows
+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     1|    296|   5.0|1147880044|
|  

## 4. Clean and join data

In [ ]:
movies = movies.dropna(subset=["movieId", "title", "genres"]).dropDuplicates(["movieId"])
ratings = ratings.dropna(subset=["userId", "movieId", "rating"])

ratings = ratings.filter((col("rating") >= 0.5) & (col("rating") <= 5.0))
data = ratings.join(movies, on="movieId", how="inner").cache()

print("Clean movies:", movies.count())
print("Clean ratings:", ratings.count())
print("Joined rows:", data.count())
data.show(5, truncate=False)

Clean movies: 62423
Clean ratings: 25000095
Joined rows: 25000095
+-------+------+------+----------+------------------------------------------------+---------------------------+
|movieId|userId|rating|timestamp |title                                           |genres                     |
+-------+------+------+----------+------------------------------------------------+---------------------------+
|296    |1     |5.0   |1147880044|Pulp Fiction (1994)                             |Comedy|Crime|Drama|Thriller|
|306    |1     |3.5   |1147868817|Three Colors: Red (Trois couleurs: Rouge) (1994)|Drama                      |
|307    |1     |5.0   |1147868828|Three Colors: Blue (Trois couleurs: Bleu) (1993)|Drama                      |
|665    |1     |5.0   |1147878820|Underground (1995)                              |Comedy|Drama|War           |
|899    |1     |3.5   |1147868510|Singin' in the Rain (1952)                      |Comedy|Musical|Romance     |
+-------+------+------+----------+----

## 5. Average rating for each movie

In [ ]:
movie_summary = data.groupBy("movieId", "title").agg(
    avg("rating").alias("average_rating"),
    count("rating").alias("number_of_ratings")
)

# At least 100 ratings makes the ranking more meaningful
top_movies = (movie_summary
              .filter(col("number_of_ratings") >= 100)
              .orderBy(desc("average_rating"), desc("number_of_ratings")))

top_movies.show(10, truncate=False)

+-------+--------------------------------+------------------+-----------------+
|movieId|title                           |average_rating    |number_of_ratings|
+-------+--------------------------------+------------------+-----------------+
|171011 |Planet Earth II (2016)          |4.483096085409253 |1124             |
|159817 |Planet Earth (2006)             |4.464796794504865 |1747             |
|318    |Shawshank Redemption, The (1994)|4.413576004516335 |81482            |
|170705 |Band of Brothers (2001)         |4.398598820058997 |1356             |
|171495 |Cosmos                          |4.3267148014440435|277              |
|858    |Godfather, The (1972)           |4.324336165187245 |52498            |
|179135 |Blue Planet II (2017)           |4.289833080424886 |659              |
|50     |Usual Suspects, The (1995)      |4.284353213163313 |55366            |
|198185 |Twin Peaks (1989)               |4.267361111111111 |288              |
|1221   |Godfather: Part II, The (1974) 

## 6. Most-rated movies

In [ ]:
most_rated = movie_summary.orderBy(desc("number_of_ratings"))
most_rated.show(10, truncate=False)

+-------+-----------------------------------------+------------------+-----------------+
|movieId|title                                    |average_rating    |number_of_ratings|
+-------+-----------------------------------------+------------------+-----------------+
|356    |Forrest Gump (1994)                      |4.048011436845787 |81491            |
|318    |Shawshank Redemption, The (1994)         |4.413576004516335 |81482            |
|296    |Pulp Fiction (1994)                      |4.188912039361382 |79672            |
|593    |Silence of the Lambs, The (1991)         |4.151341616415071 |74127            |
|2571   |Matrix, The (1999)                       |4.154099127610975 |72674            |
|260    |Star Wars: Episode IV - A New Hope (1977)|4.120188599618726 |68717            |
|480    |Jurassic Park (1993)                     |3.6791749812920926|64144            |
|527    |Schindler's List (1993)                  |4.247579083279535 |60411            |
|110    |Braveheart (

## 7. Average rating by genre

In [ ]:
genre_data = data.withColumn("genre", explode(split(col("genres"), r"\\|")))

genre_summary = genre_data.groupBy("genre").agg(
    avg("rating").alias("average_rating"),
    count("rating").alias("rating_count")
).orderBy(desc("average_rating"))

genre_summary.show(truncate=False)

+-----+------------------+------------+
|genre|average_rating    |rating_count|
+-----+------------------+------------+
|N    |3.9257281769386028|247227      |
|W    |3.7346387394729073|1751077     |
|D    |3.677987444177292 |11285282    |
|a    |3.6328445896727706|36271161    |
|M    |3.6251061649720713|4038526     |
|X    |3.6037121959523324|1063279     |
|I    |3.6037121959523324|1063279     |
|m    |3.584165213321673 |30777276    |
|s    |3.5725148597445355|6343817     |
|r    |3.558598847658693 |46671242    |
||    |3.548307991038779 |42809791    |
|R    |3.5427121571630567|4497291     |
|     |3.533854451353085 |25000095    |
|u    |3.53100057942514  |7119125     |
|i    |3.528977999730806 |33923494    |
|t    |3.525187780692204 |20585716    |
|e    |3.523634457359019 |41546945    |
|T    |3.5229643285084498|6763272     |
|l    |3.5189855377268917|16888908    |
|v    |3.517444633654892 |5832424     |
+-----+------------------+------------+
only showing top 20 rows


## 8. Rating distribution

In [ ]:
rating_summary = ratings.groupBy("rating").agg(
    count("*").alias("count")
).orderBy("rating")

rating_summary.show()

+------+-------+
|rating|  count|
+------+-------+
|   0.5| 393068|
|   1.0| 776815|
|   1.5| 399490|
|   2.0|1640868|
|   2.5|1262797|
|   3.0|4896928|
|   3.5|3177318|
|   4.0|6639798|
|   4.5|2200539|
|   5.0|3612474|
+------+-------+



## 9. Save results as CSV

In [ ]:
output_dir = "/content/movie_analysis_results"
os.makedirs(output_dir, exist_ok=True)

top_movies.coalesce(1).write.mode("overwrite").option("header", True).csv(
    output_dir + "/top_movies"
)
most_rated.limit(100).coalesce(1).write.mode("overwrite").option("header", True).csv(
    output_dir + "/most_rated_movies"
)
genre_summary.coalesce(1).write.mode("overwrite").option("header", True).csv(
    output_dir + "/genre_summary"
)
rating_summary.coalesce(1).write.mode("overwrite").option("header", True).csv(
    output_dir + "/rating_distribution"
)

print("CSV result folders saved in:", output_dir)

CSV result folders saved in: /content/movie_analysis_results


## 10. Optional: download all results as one ZIP

In [ ]:
import shutil
from google.colab import files

zip_results = shutil.make_archive("/content/movie_analysis_results", "zip", output_dir)
files.download(zip_results)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusion
This project uses PySpark to load and process the MovieLens 25M ratings dataset. It demonstrates data loading, cleaning, joins, groupBy aggregations, sorting, genre analysis, and saving results.

**Note:** Spark runs in local mode on Colab. This demonstrates Apache Spark; it does not configure Hadoop HDFS or Cassandra.